In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
"""Question 2"""

import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [4]:
question = "How does the agentic loop keep calling the model until it stops?"
results = index.search(
    question,
    num_results=5
    # boost_dict={"question": 2.0, "section": 0.5}
)

In [5]:
from rag_helper import RAGBase

class LessonRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        answer = response.output_text
        usage = response.usage

        return answer, usage

In [6]:
from openai import OpenAI

openai_client = OpenAI()

rag = LessonRAG(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

query = "How does the agentic loop keep calling the model until it stops?"

answer, usage = rag.rag(query)

print(answer)
print(usage.input_tokens)

It keeps calling the model inside a `while True` loop.

After each model response, the code checks whether the response contains any `function_call` items:

- If there are function calls, it runs the tools, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, the agent keeps going until the model returns a final message with no more tool calls.
7126


In [ ]:
# Question 4
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [9]:
# Question 5
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [10]:
from openai import OpenAI

openai_client = OpenAI()

rag = LessonRAG(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

query = "How does the agentic loop keep calling the model until it stops?"

answer, usage = rag.rag(query)

print(answer)
print(usage.input_tokens)

It keeps calling the model inside a `while True` loop and checks whether the model made any `function_call`s on that turn.

- If there is at least one function call, the code runs the tool, appends the result to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and returns the final answer.

So the stop condition is: **no function calls this iteration**.
2309


In [11]:
def search(query: str) -> list[dict]:
    """
    Search the course lesson chunks for information related to the query.
    """
    results = index.search(
        query,
        num_results=5
    )

    return results

In [12]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course lesson chunks for information related to the query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course lesson chunks."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [13]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [14]:
def agent_loop(instructions, question, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    search_call_count = 0
    last_answer = None
    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)

                if item.name == "search":
                    search_call_count += 1

                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(last_answer)

        it = it + 1

        if has_function_calls == False:
            break

    return last_answer, search_call_count

In [15]:
instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

question = "How does the agentic loop work, and how is it different from plain RAG?"

answer, search_call_count = agent_loop(
    instructions=instructions,
    question=question,
    model="gpt-5.4-mini"
)

print("Search calls:", search_call_count)

iteration #1...
function_call: search {"query":"agentic loop plain RAG difference retrieval generation tool use iterative reasoning course lesson"}
function_call: search {"query":"agentic loop how it works RAG course lesson chunks agentic workflow"}
function_call: search {"query":"plain RAG vs agentic loop retrieval augmented generation iterative loop tools"}
function_call: search {"query":"agentic loop decides next action retrieve generate reflect course"}
iteration #2...
ASSISTANT:
The **agentic loop** is a repeated cycle where the model can decide what to do next, instead of you hard-coding a single retrieve-then-answer step.

### How the agentic loop works
From the course, the core pattern is:

1. Send the user question to the LLM.
2. The LLM may return a **tool call** instead of a final answer.
3. Your code executes that tool call.
4. You send the tool result back to the LLM.
5. Repeat until the model returns a final answer with **no more tool calls**.

In pseudocode, it looks lik